# Lesson 02 — 让一次按下只算一次 — 参考实现

先完成主 Notebook 的练习再阅读。此 Notebook 只展示可通过 exercise 契约的一种实现。


In [ ]:
# 运行支架：只运行一次，不需要学习或修改 Python。
from pathlib import Path
project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "tools/notebook_setup.py").is_file()), None)
if project is None:
    raise RuntimeError("请用 VS Code 打开 esp32-hands-on-academy 文件夹")
get_ipython().run_line_magic("run", str(project / "tools/notebook_setup.py"))


运行代码格，使用同一个可交互面板验证。然后回到主 Notebook，对照自己的状态和边界处理。


In [ ]:
%%academy_lab 02-reliable-button solution exercise pass
#pragma once
#include "academy/device.hpp"
#include <cstdint>
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& board);
        void tick();

    private:
        Device& board_;
        bool initialized_ = false;
        bool raw_ = false;
        bool stable_ = false;
        std::uint32_t changed_at_ = 0;
    };
}

namespace academy {
    Controller::Controller(Device& board) : board_(board) {
    }
    void Controller::tick() {
        const bool pressed = board_.read_button() == Level::low;
        if (!initialized_) {
            initialized_ = true;
            raw_ = stable_ = pressed;
            changed_at_ = board_.input.now_ms;
            board_.output.state = "baseline";
            return;
        }
        if (pressed != raw_) {
            raw_ = pressed;
            changed_at_ = board_.input.now_ms;
        }
        if (raw_ != stable_ && board_.input.now_ms - changed_at_ >= 30) {
            stable_ = raw_;
            if (stable_) {
                ++board_.output.events;
                board_.output.led = !board_.output.led;
            }
        }
        board_.output.state = stable_ ? "stable pressed" : "stable released";
    }
}
